# Deploy the UC-registered RAG agent to a Mosaic AI Model Serving CPU endpoint, configure scale-to-zero, and confirm inference tables capture every query

The chain is registered, the platform team wants it behind a REST endpoint so the customer-support web app can hit it from a browser. You have one session to ship one: scale-to-zero so the cost stops between requests, inference tables so compliance can audit, three test queries to prove the round-trip works.

The hands-on work:

- Create a Mosaic AI Model Serving endpoint named `labrun-rag-endpoint` that serves the UC-registered champion alias from Lab 8.
- Configure `scale_to_zero_enabled=True` and `workload_size='Small'` (Free Edition's binding constraint is the active-endpoint count).
- Configure `auto_capture_config` so every request and response is written to a Delta inference table at `workspace.default.labrun_rag_endpoint_payload`.
- Send three test queries via `WorkspaceClient.serving_endpoints.query(...)`, wait for the inference table to flush, and confirm the rows landed.

Estimated time: 70 minutes (endpoint provisioning is the long pole; budget 5 to 15 minutes for the first READY transition). Cost: $0.00 to $0.05 (Free Edition includes the endpoint compute; the FM API token spend is fractions of a cent).


In [ ]:
# NBVAL_SKIP
# Pinned dependencies. The inference call uses the same SDK as the rest of the track.

!pip install --quiet labrun-checks==0.7.0 databricks-sdk==0.40.0 mlflow==2.20.0 langchain==0.3.7 databricks-langchain==0.1.1


In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. UC identifiers (schema, table, volume) use
# snake_case under the labrun_ prefix because Unity Catalog identifiers prefer
# snake_case (hyphens force backtick-quoting everywhere downstream).

import atexit
import getpass
import json
import re
import sys
import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "mosaic-ai-model-serving-rag-endpoint"
LAB_TAG_KEY = "labrun_lab_slug"
LAB_TAG_VALUE = LAB_ID  # must equal cert YAML labs[].slug exactly

CATALOG = "workspace"
PARENT_SCHEMA = "default"
LAB_SCHEMA = "default"
LAB_SCHEMA_FQN = f"{CATALOG}.{PARENT_SCHEMA}.{LAB_SCHEMA}"

SERVING_ENDPOINT_NAME = "labrun-rag-endpoint"
REGISTERED_MODEL_NAME = "workspace.default.labrun_genai_pyfunc.rag_agent"
CHAMPION_ALIAS = "champion"
INFERENCE_TABLE_FQN = f"{CATALOG}.{PARENT_SCHEMA}.labrun_rag_endpoint_payload"


In [ ]:
# NBVAL_SKIP
# Register the session, validate Databricks credentials via the SDK, and
# resolve the Starter SQL warehouse. This cell must succeed before the
# manifest cell runs anything.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
databricks_host = getpass.getpass("Databricks workspace URL (https://...azuredatabricks.net or .cloud.databricks.com): ").strip()
databricks_token = getpass.getpass("Databricks personal access token (PAT): ").strip()

if not databricks_host or not databricks_token:
    print("Workspace URL and PAT are both required. Re-run this cell.")
    raise SystemExit(1)
if not databricks_host.startswith("https://"):
    databricks_host = f"https://{databricks_host}"

w = WorkspaceClient(host=databricks_host, token=databricks_token)

try:
    me = w.current_user.me()
except Exception as exc:
    print("Databricks credentials are missing or expired. CurrentUser.me() failed:")
    print(f"  {exc}")
    print()
    print("Refresh your workspace URL and PAT, then re-run this cell.")
    raise SystemExit(1)

CALLER_USER = me.user_name or (me.emails[0].value if me.emails else "unknown")
print(f"Credentials validated. Workspace user: {CALLER_USER}")

warehouses = list(w.warehouses.list())
if not warehouses:
    print("No SQL warehouses found in this workspace. Free Edition ships with")
    print("a Starter warehouse by default; if it has been deleted, recreate it")
    print("from the SQL > Warehouses page before re-running this cell.")
    raise SystemExit(1)
WAREHOUSE = warehouses[0]
WAREHOUSE_ID = WAREHOUSE.id
print(f"SQL warehouse resolved: {WAREHOUSE.name} ({WAREHOUSE_ID})")

DATABRICKS_CREDENTIALS = {
    "host": databricks_host,
    "token": databricks_token,
    "warehouse_id": WAREHOUSE_ID,
}


def run_sql(statement, warehouse_id=None, wait_seconds=180):
    """Submit SQL to the warehouse and return {columns, rows}."""
    wid = warehouse_id or WAREHOUSE_ID
    resp = w.statement_execution.execute_statement(
        warehouse_id=wid, statement=statement, wait_timeout="50s"
    )
    statement_id = resp.statement_id
    deadline = time.time() + wait_seconds
    while True:
        state = resp.status.state if resp.status else None
        if state == StatementState.SUCCEEDED:
            break
        if state in (StatementState.FAILED, StatementState.CANCELED, StatementState.CLOSED):
            err = resp.status.error.message if (resp.status and resp.status.error) else "no error message"
            raise RuntimeError(f"SQL failed ({state}): {err}\n  Statement: {statement}")
        if time.time() > deadline:
            raise TimeoutError(
                f"SQL did not finish in {wait_seconds}s. Last state: {state}. "
                f"The Starter warehouse may still be waking up; re-run the cell."
            )
        time.sleep(2)
        resp = w.statement_execution.get_statement(statement_id)
    columns = []
    if resp.manifest and resp.manifest.schema and resp.manifest.schema.columns:
        columns = [c.name for c in resp.manifest.schema.columns]
    rows = []
    if resp.result and resp.result.data_array:
        rows = list(resp.result.data_array)
    return {"columns": columns, "rows": rows}


register(session_token=session_token, lab_id=LAB_ID, credentials=DATABRICKS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")


In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# Manifest is module-level and in reverse-creation order. Per
# RESOURCE-SAFETY-SPEC Rule 4 the orphan scan blocks execution if any
# tagged resources from a prior session exist.

USER_EMAIL = CALLER_USER

CLEANUP_MANIFEST = [
    CleanupResource(
        type="model_serving_endpoint",
        id=SERVING_ENDPOINT_NAME,
        region="databricks",
        cli_delete_command=f"databricks serving-endpoints delete {SERVING_ENDPOINT_NAME}",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=INFERENCE_TABLE_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {INFERENCE_TABLE_FQN}\"",
    ),
]


class DatabricksCleanupAdapter:
    """Inline adapter for UC + MLflow + serving + vector search resources."""

    def delete_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "uc_managed_table":
            run_sql(f"DROP TABLE IF EXISTS {rid}")
        elif rtype == "uc_view":
            run_sql(f"DROP VIEW IF EXISTS {rid}")
        elif rtype == "uc_volume":
            run_sql(f"DROP VOLUME IF EXISTS {rid}")
        elif rtype == "uc_volume_contents":
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                listing = list(w.files.list_directory_contents(volume_path))
            except Exception:
                return
            for entry in listing:
                try:
                    if entry.is_directory:
                        w.files.delete_directory(entry.path)
                    else:
                        w.files.delete(entry.path)
                except Exception:
                    pass
        elif rtype == "uc_schema":
            run_sql(f"DROP SCHEMA IF EXISTS {rid} CASCADE")
        elif rtype == "uc_registered_model":
            try:
                import mlflow
                mlflow.set_registry_uri("databricks-uc")
                mlflow.MlflowClient().delete_registered_model(name=rid)
            except Exception:
                pass
        elif rtype == "mlflow_experiment":
            try:
                import mlflow
                exp = mlflow.get_experiment_by_name(rid)
                if exp is not None:
                    mlflow.delete_experiment(exp.experiment_id)
            except Exception:
                pass
        elif rtype == "model_serving_endpoint":
            try:
                w.serving_endpoints.delete(name=rid)
                deadline = time.time() + 600
                while time.time() < deadline:
                    try:
                        w.serving_endpoints.get(name=rid)
                    except Exception:
                        return
                    time.sleep(5)
            except Exception:
                pass
        elif rtype == "vector_search_index":
            try:
                w.vector_search_indexes.delete_index(index_name=rid)
            except Exception:
                pass
        elif rtype == "vector_search_endpoint":
            try:
                w.vector_search_endpoints.delete_endpoint(endpoint_name=rid)
            except Exception:
                pass
        else:
            raise ValueError(f"DatabricksCleanupAdapter: unknown resource type {rtype!r}")

    def describe_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype in ("uc_managed_table", "uc_view"):
            parts = rid.split(".")
            if len(parts) != 3:
                return False
            catalog, schema, table = parts
            try:
                sql = (
                    "SELECT 1 FROM system.information_schema.tables "
                    f"WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' "
                    f"AND table_name = '{table}'"
                )
                return len(run_sql(sql)["rows"]) > 0
            except Exception:
                return False
        if rtype == "uc_volume":
            parts = rid.split(".")
            if len(parts) != 3:
                return False
            catalog, schema, volume = parts
            try:
                sql = (
                    "SELECT 1 FROM system.information_schema.volumes "
                    f"WHERE volume_catalog = '{catalog}' AND volume_schema = '{schema}' "
                    f"AND volume_name = '{volume}'"
                )
                return len(run_sql(sql)["rows"]) > 0
            except Exception:
                return False
        if rtype == "uc_volume_contents":
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                listing = list(w.files.list_directory_contents(volume_path))
                return len(listing) > 0
            except Exception:
                return False
        if rtype == "uc_schema":
            parts = rid.split(".")
            if len(parts) != 2:
                return False
            catalog, schema = parts
            try:
                sql = (
                    "SELECT 1 FROM system.information_schema.schemata "
                    f"WHERE catalog_name = '{catalog}' AND schema_name = '{schema}'"
                )
                return len(run_sql(sql)["rows"]) > 0
            except Exception:
                return False
        if rtype == "uc_registered_model":
            try:
                import mlflow
                mlflow.set_registry_uri("databricks-uc")
                mlflow.MlflowClient().get_registered_model(name=rid)
                return True
            except Exception:
                return False
        if rtype == "mlflow_experiment":
            try:
                import mlflow
                return mlflow.get_experiment_by_name(rid) is not None
            except Exception:
                return False
        if rtype == "model_serving_endpoint":
            try:
                w.serving_endpoints.get(name=rid)
                return True
            except Exception:
                return False
        if rtype == "vector_search_index":
            try:
                w.vector_search_indexes.get_index(index_name=rid)
                return True
            except Exception:
                return False
        if rtype == "vector_search_endpoint":
            try:
                w.vector_search_endpoints.get_endpoint(endpoint_name=rid)
                return True
            except Exception:
                return False
        return False

    def tag_scan(self, credentials, lab_slug, region):
        arns = []
        queries = [
            (
                "SELECT catalog_name, schema_name FROM system.information_schema.schema_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}",
            ),
            (
                "SELECT catalog_name, schema_name, table_name FROM system.information_schema.table_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
        ]
        for sql, fmt in queries:
            try:
                result = run_sql(sql)
            except Exception:
                continue
            for row in result["rows"]:
                arns.append(fmt(row))
        return arns


CLEANUP_ADAPTER = DatabricksCleanupAdapter()


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans():
    return CLEANUP_ADAPTER.tag_scan(DATABRICKS_CREDENTIALS, LAB_TAG_VALUE, "databricks")


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing UC objects tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom first, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")


## Resolve the champion alias and confirm the prerequisite

The serving endpoint deploys the version that `workspace.default.labrun_genai_pyfunc.rag_agent@champion` resolves to today. If the alias is unset (Lab 8 was not run, or the model was dropped), the endpoint create call fails with an unhelpful error. This cell resolves the alias up front and prints the version so the operator sees what is about to deploy.


In [ ]:
# NBVAL_SKIP
# Resolve the champion alias version before the endpoint create call.

import mlflow
mlflow.set_registry_uri("databricks-uc")
_client = mlflow.MlflowClient()

try:
    CHAMPION_VERSION = _client.get_model_version_by_alias(
        name=REGISTERED_MODEL_NAME, alias=CHAMPION_ALIAS
    ).version
except Exception as exc:
    print(
        f"Prerequisite missing: alias {CHAMPION_ALIAS!r} on "
        f"{REGISTERED_MODEL_NAME} did not resolve. Run Lab 8 first "
        f"or re-set the alias. Error: {exc!r}"
    )
    raise SystemExit(1)

print(f"Champion version resolved: {CHAMPION_VERSION}")


## Task 1: Create the serving endpoint with scale-to-zero and inference tables

Two SDK building blocks:

- `EndpointCoreConfigInput` with a `served_entities` list pointing at the three-level UC model name and the champion version.
- `AutoCaptureConfigInput` with `catalog_name='workspace'`, `schema_name='default'`, `table_name_prefix='labrun_rag_endpoint'`, `enabled=True`. The canonical inference table path is `<catalog>.<schema>.<table_name_prefix>_payload`.

The create call returns immediately; the endpoint transitions to READY over 5 to 15 minutes the first time. Poll `serving_endpoints.get(name=...)` with a 900-second ceiling. The checkpoint validates the READY state plus the served entity name and version.


In [ ]:
# NBVAL_SKIP
# Task 1: create the serving endpoint, wait for READY.

from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
    AutoCaptureConfigInput,
)

config = EndpointCoreConfigInput(
    served_entities=[
        ServedEntityInput(
            entity_name=REGISTERED_MODEL_NAME,
            entity_version=CHAMPION_VERSION,
            workload_size="Small",
            scale_to_zero_enabled=True,
        )
    ],
    auto_capture_config=AutoCaptureConfigInput(
        catalog_name=CATALOG,
        schema_name=PARENT_SCHEMA,
        table_name_prefix="labrun_rag_endpoint",
        enabled=True,
    ),
)

# YOUR CODE: try fetching the endpoint; if it does not exist, create it via
# YOUR CODE:   w.serving_endpoints.create(name=SERVING_ENDPOINT_NAME, config=config)
# YOUR CODE: poll w.serving_endpoints.get(name=...) every 30 seconds with a
# YOUR CODE:   900-second deadline; break when ep.state.ready == "READY"
# YOUR CODE: print the elapsed wait time so the operator sees the cold-start cost


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: endpoint exists, is READY, scale_to_zero is enabled, served
# entity is the champion alias's version on workload_size=Small.


def checkpoint_1(session):
    try:
        ep = w.serving_endpoints.get(name=SERVING_ENDPOINT_NAME)
    except Exception as exc:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Endpoint {SERVING_ENDPOINT_NAME!r} not found: {exc!r}. "
                f"Did serving_endpoints.create() run?"
            ),
        )
    state = getattr(ep.state, "ready", None)
    state_value = getattr(state, "value", state) if state is not None else None
    if str(state_value) != "READY":
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Endpoint state is {state_value!r}; expected READY. "
                f"Provisioning takes 5 to 15 minutes; re-run Task 1's wait loop."
            ),
        )
    served = ep.config.served_entities[0] if ep.config and ep.config.served_entities else None
    if served is None:
        return CheckpointResult(
            status="fail",
            error_reason="Endpoint has no served_entities.",
        )
    if served.entity_name != REGISTERED_MODEL_NAME:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Endpoint serves {served.entity_name!r}; expected {REGISTERED_MODEL_NAME!r}."
            ),
        )
    if str(served.entity_version) != str(CHAMPION_VERSION):
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Endpoint serves version {served.entity_version}; "
                f"expected champion version {CHAMPION_VERSION}."
            ),
        )
    if not served.scale_to_zero_enabled:
        return CheckpointResult(
            status="fail",
            error_reason=(
                "scale_to_zero_enabled is False on the served entity. "
                "Free Edition's binding constraint is the active-endpoint count, "
                "not cost, but the test still requires scale-to-zero per the brief."
            ),
        )
    return CheckpointResult(status="pass")


check(1, checkpoint_1)


<details><summary>Hint 1 (nudge)</summary>

The pattern is create-or-get plus poll. Wrap the create in a try-except: if it raises because the endpoint already exists, fall through to the get. Then poll until READY.

</details>

<details><summary>Hint 2 (direction)</summary>

```
try:
    w.serving_endpoints.get(name=SERVING_ENDPOINT_NAME)
    print("Endpoint already exists; waiting for READY.")
except Exception:
    print("Creating endpoint...")
    w.serving_endpoints.create(name=SERVING_ENDPOINT_NAME, config=config)
```

Then poll: `ep.state.ready == "READY"` (or `ep.state.ready.value == "READY"` depending on the SDK version). `time.sleep(30)` between polls. Cap at 900 seconds.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
try:
    w.serving_endpoints.get(name=SERVING_ENDPOINT_NAME)
    print(f"Endpoint {SERVING_ENDPOINT_NAME} already exists; waiting for READY...")
except Exception:
    print(f"Creating endpoint {SERVING_ENDPOINT_NAME}...")
    w.serving_endpoints.create(name=SERVING_ENDPOINT_NAME, config=config)

deadline = time.time() + 900
start = time.time()
while time.time() < deadline:
    ep = w.serving_endpoints.get(name=SERVING_ENDPOINT_NAME)
    state = getattr(ep.state, "ready", None)
    state_value = getattr(state, "value", state) if state is not None else None
    print(f"  {int(time.time() - start)}s: state={state_value}")
    if str(state_value) == "READY":
        break
    time.sleep(30)
else:
    raise TimeoutError("Endpoint did not reach READY in 900s")

print(f"READY in {int(time.time() - start)}s.")
```

</details>


**Wallet check.** Endpoint provisioning on Free Edition is included in the daily compute quota. No dollar bill yet. The pole here is wall-clock: 5 to 15 minutes of polling while the platform pulls the artifact, builds the image, and warms a CPU container.


## Task 2: Confirm the inference table was created

The Mosaic AI auto-capture pipeline creates the inference table the first time a request lands on the endpoint, not at endpoint-create time. So either send a probe query first OR confirm the table appears within a minute of the first request.

The canonical schema for the inference table includes `request_id`, `request_time`, `request`, `response`, `status_code`. Some versions also include `databricks_request_id` and `request_metadata`. The checkpoint asserts the table exists and that the request and response columns are present.


In [ ]:
# NBVAL_SKIP
# Task 2: kick off one probe query so the auto-capture pipeline materializes
# the inference table, then poll until information_schema sees it.

# YOUR CODE: call w.serving_endpoints.query(
#     name=SERVING_ENDPOINT_NAME, inputs=["warmup probe"]
# ) inside a try-except so a cold-start blip does not stop the lab
# YOUR CODE: poll system.information_schema.tables for INFERENCE_TABLE_FQN
# YOUR CODE:   sleep 15 seconds between polls, ceiling 300 seconds
# YOUR CODE: print when the inference table first appears


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: inference table exists at workspace.default.labrun_rag_endpoint_payload
# with the columns the auto-capture pipeline writes.


def checkpoint_2(session):
    catalog, schema, table = INFERENCE_TABLE_FQN.split(".")
    try:
        rows = run_sql(
            "SELECT column_name FROM system.information_schema.columns "
            f"WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' "
            f"AND table_name = '{table}'"
        )["rows"]
    except Exception as exc:
        return CheckpointResult(
            status="error",
            error_reason=f"Could not read columns for {INFERENCE_TABLE_FQN}: {exc!r}",
        )
    if not rows:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Inference table {INFERENCE_TABLE_FQN} not found. The auto-capture "
                f"pipeline materializes the table after the first request lands; "
                f"send a probe query and wait 1 to 5 minutes."
            ),
        )
    columns = {r[0].lower() for r in rows}
    for required in ("request", "response"):
        if required not in columns:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Inference table missing column {required!r}. "
                    f"Observed: {sorted(columns)}"
                ),
            )
    return CheckpointResult(status="pass")


check(2, checkpoint_2)


<details><summary>Hint 1 (nudge)</summary>

The auto-capture pipeline is lazy. The table does not exist until the first request lands. Send a probe query, then poll information_schema for the table to appear.

</details>

<details><summary>Hint 2 (direction)</summary>

```
try:
    w.serving_endpoints.query(name=SERVING_ENDPOINT_NAME, inputs=["warmup probe"])
except Exception as exc:
    print(f"Probe query raised (cold-start is normal): {exc!r}")

catalog, schema, table = INFERENCE_TABLE_FQN.split(".")
deadline = time.time() + 300
while time.time() < deadline:
    rows = run_sql(
        f"SELECT 1 FROM system.information_schema.tables "
        f"WHERE table_catalog='{catalog}' AND table_schema='{schema}' AND table_name='{table}'"
    )["rows"]
    if rows:
        break
    time.sleep(15)
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
try:
    w.serving_endpoints.query(name=SERVING_ENDPOINT_NAME, inputs=["warmup probe"])
    print("Probe query sent.")
except Exception as exc:
    print(f"Probe query raised (cold-start is normal): {exc!r}")

catalog, schema, table = INFERENCE_TABLE_FQN.split(".")
deadline = time.time() + 300
while time.time() < deadline:
    rows = run_sql(
        f"SELECT 1 FROM system.information_schema.tables "
        f"WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' "
        f"AND table_name = '{table}'"
    )["rows"]
    if rows:
        print(f"Inference table {INFERENCE_TABLE_FQN} now visible.")
        break
    time.sleep(15)
else:
    raise TimeoutError("Inference table did not appear in 300s")
```

</details>


**Wallet check.** One probe query plus information_schema polls. The probe call cold-started the endpoint, which means the first response was a 30 to 90 second wall-clock spend. Tokens billed: about 300. Cost: under a cent.


## Task 3: Send three test queries and capture responses

Three lab-provided fixtures. Send each via `serving_endpoints.query(name=..., inputs=[...])`, capture the response, and print a single-line summary so the operator sees the round-trip in the cell output.

The endpoint may scale to zero between requests; the first query in a warm window is fast, the first query after a cold window is 30 to 90 seconds.


In [ ]:
# NBVAL_SKIP
# Task 3: three serving_endpoints.query() calls against the endpoint.

TEST_QUERIES = [
    "How do I sign up for Databricks Free Edition?",
    "What is Mosaic AI Vector Search Delta-Sync?",
    "Explain the three-level naming in Unity Catalog.",
]

responses = []
# YOUR CODE: for each query in TEST_QUERIES, call w.serving_endpoints.query(
#     name=SERVING_ENDPOINT_NAME, inputs=[query])
# YOUR CODE: extract the response text. The pyfunc signature returns array<string>
#     so the result is at resp.predictions[0]. Append (query, response) to responses.
# YOUR CODE: print a one-line summary per query so the operator sees the round-trip.


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: three responses captured, all non-empty.


def checkpoint_3(session):
    if not isinstance(responses, list) or len(responses) != 3:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"responses list has {len(responses) if isinstance(responses, list) else 'no'}"
                f" items; expected exactly 3."
            ),
        )
    for idx, item in enumerate(responses):
        if not isinstance(item, tuple) or len(item) != 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"responses[{idx}] is {item!r}; expected (query, response) tuple."
                ),
            )
        query, resp = item
        if not isinstance(resp, str) or not resp.strip():
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Query {query!r} returned empty response. "
                    f"Endpoint may have cold-started; re-run Task 3."
                ),
            )
    return CheckpointResult(status="pass")


check(3, checkpoint_3)


<details><summary>Hint 1 (nudge)</summary>

`w.serving_endpoints.query(name=..., inputs=[query])` returns a `QueryEndpointResponse`. For pyfunc models whose signature outputs array of string, the answer is in `resp.predictions[0]` (a single-element list, take index 0).

</details>

<details><summary>Hint 2 (direction)</summary>

```
for query in TEST_QUERIES:
    resp = w.serving_endpoints.query(name=SERVING_ENDPOINT_NAME, inputs=[query])
    text = resp.predictions[0]
    responses.append((query, text))
    print(f"  {query!r} -> {text[:80]!r}")
```

If `resp.predictions` is not the right attribute on this SDK version, inspect the response with `print(resp.__dict__)` for the available fields.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
for query in TEST_QUERIES:
    resp = w.serving_endpoints.query(
        name=SERVING_ENDPOINT_NAME,
        inputs=[query],
    )
    text = resp.predictions[0] if isinstance(resp.predictions[0], str) else str(resp.predictions[0])
    responses.append((query, text))
    print(f"  {query!r}")
    print(f"    -> {text[:120]!r}")
```

</details>


**Wallet check.** Three chain invocations: each is one LLM call (about 300 tokens) plus a quick retriever lookup. Roughly 1000 tokens billed at fractions of a cent. The endpoint stays warm during the lab session, so the second and third queries return in under a second.


## Task 4: Confirm the inference table captured every request

The auto-capture pipeline flushes asynchronously. Typical lag is 1 to 5 minutes on Free Edition. The checkpoint polls with a 600-second ceiling and asserts that the table has at least three rows with non-null request and response columns matching the Task 3 traffic.


In [ ]:
# NBVAL_SKIP
# Task 4: wait for the inference table to flush, then verify rows landed.

# YOUR CODE: poll SELECT count(*) FROM INFERENCE_TABLE_FQN every 20 seconds
# YOUR CODE: stop when the count is >= 3 (the three Task 3 queries plus the probe)
# YOUR CODE: ceiling of 600 seconds; print the wait progress
# YOUR CODE: when ready, SELECT request, response FROM INFERENCE_TABLE_FQN LIMIT 3
# YOUR CODE:   and print the first 100 characters of each row's request payload.


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: inference table contains rows for every Task 3 query and
# request/response are non-null.


def checkpoint_4(session):
    deadline = time.time() + 600
    while time.time() < deadline:
        try:
            count = int(run_sql(
                f"SELECT count(*) FROM {INFERENCE_TABLE_FQN} "
                f"WHERE request IS NOT NULL AND response IS NOT NULL"
            )["rows"][0][0])
        except Exception as exc:
            return CheckpointResult(
                status="error",
                error_reason=(
                    f"Could not query {INFERENCE_TABLE_FQN}: {exc!r}"
                ),
            )
        if count >= 3:
            return CheckpointResult(status="pass")
        time.sleep(20)
    return CheckpointResult(
        status="fail",
        error_reason=(
            f"Inference table only has {count} captured rows after 600s wait. "
            f"The auto-capture pipeline typically flushes within 1 to 5 minutes; "
            f"if no rows ever appear, confirm auto_capture_config.enabled=True on the endpoint."
        ),
    )


check(4, checkpoint_4)


<details><summary>Hint 1 (nudge)</summary>

The inference-table writer is async. Sleep, count, repeat. The fail message names the symptom and the remediation: `auto_capture_config.enabled` must be True for any rows to land.

</details>

<details><summary>Hint 2 (direction)</summary>

```
deadline = time.time() + 600
while time.time() < deadline:
    count = int(run_sql(f"SELECT count(*) FROM {INFERENCE_TABLE_FQN}")["rows"][0][0])
    print(f"  rows so far: {count}")
    if count >= 3:
        break
    time.sleep(20)
```

After the loop, run a SELECT and print three rows' request payloads (the column holds a JSON-encoded body).

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4:

```python
deadline = time.time() + 600
final_count = 0
while time.time() < deadline:
    final_count = int(run_sql(
        f"SELECT count(*) FROM {INFERENCE_TABLE_FQN} "
        f"WHERE request IS NOT NULL AND response IS NOT NULL"
    )["rows"][0][0])
    print(f"  inference rows so far: {final_count}")
    if final_count >= 3:
        break
    time.sleep(20)

rows = run_sql(
    f"SELECT request, response FROM {INFERENCE_TABLE_FQN} LIMIT 3"
)["rows"]
for req, resp in rows:
    print(f"  REQ: {str(req)[:100]!r}")
    print(f"  RES: {str(resp)[:100]!r}")
    print()
```

</details>


**Wallet check.** Polling against information_schema and the inference table itself touches the Starter warehouse for a fraction of a warehouse-second. The expensive cell in the lab was Task 1's provisioning wait, which Free Edition swallows. Total dollar bill: still under five cents.


## Cleanup

Tear down the serving endpoint first (releases the active-endpoint slot on Free Edition; without this Lab 10 cannot stand up its own serving endpoint), then drop the inference table. The endpoint delete waits for the platform to confirm full deletion (typical 30 to 90 seconds) so a re-run of this lab does not collide with a still-deleting endpoint name.

Lab 10 calls this endpoint as its evaluator target. If you are going straight to Lab 10, skip the cleanup cell here and run Lab 10 first; Lab 10's setup checks for the endpoint and Lab 10's cleanup cleans up both. If you are stopping after Lab 9, run the cleanup cell below.


In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order using the inline Databricks adapter. Prints the canonical summary
# from RESOURCE-SAFETY-SPEC Rule 6 and sys.exit(1) on any failure.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Databricks workspace may still hold lab objects. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)


**Session total: about $0.03.** One probe plus three test queries plus three Task 4 round-trip prints. Roughly 1300 tokens billed across the FM API. Free Edition swallowed the endpoint compute. Cleanup tore down the serving endpoint and dropped the inference table; the active-endpoint slot is free again for Lab 10.


## Reflection

These are not graded. They are for you.

1. Walk through what happens when a query arrives at a scale-to-zero serving endpoint that has been idle for 30 minutes. Name each step (cold-start, container spin-up, dependency import, model load via `load_context`, first inference). Where is the latency dominated, and what cuts that latency for a production high-traffic endpoint (the answer is on the Premium side)?

2. The inference table captures every request and response. A compliance auditor wants to know: how would you redact PII from inference table rows BEFORE the auditor reads them? Walk through the architecture: the regex you would run, where you would run it, what schema change the redacted table would have versus the raw one.

## Exam-Style Practice

**Question 1 (Domain 4, serving endpoint configuration):**

A GenAI engineer is deploying a RAG agent to Mosaic AI Model Serving. The application sees variable traffic (zero queries overnight, 50 per minute during business hours). Cost matters. Which serving configuration is the best fit?

A. `scale_to_zero_enabled=False`, `min_provisioned_throughput=10` (always-on with reserved capacity).
B. `scale_to_zero_enabled=True`, with auto-scaling to peak when traffic arrives.
C. A GPU-backed endpoint with provisioned throughput.
D. Three separate endpoints (one for night, one for morning, one for evening) with traffic split.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: always-on bills for compute even at zero traffic. Reserved capacity is the wrong shape for bursty traffic.
- B is correct: scale-to-zero plus auto-scaling matches bursty traffic at minimal cost. The first-query cold-start (30 to 90 seconds) is the trade-off; if the application can tolerate it, this is the cheapest correct answer.
- C is wrong: GPU is not needed for RAG inference. The LLM call goes out to the Foundation Model API, which is GPU-backed on Databricks side; the serving endpoint here is the chain orchestrator and runs fine on CPU. Provisioned throughput is for high-volume always-on workloads.
- D is wrong: three endpoints is over-engineered and Free Edition's endpoint cap blocks it anyway.

</details>

**Question 2 (Domain 6, inference table usage):**

A GenAI engineer needs to debug why their RAG endpoint occasionally returns "I do not know" instead of a useful answer. The endpoint has inference tables enabled. Which approach is the fastest path to root cause?

A. Re-run the same queries manually and watch the chain trace in the notebook.
B. Query the inference table for rows where the response contains "I do not know" and inspect the corresponding request bodies, retrieved chunk IDs, and chain trace.
C. Add more `print()` statements to the chain code and redeploy.
D. Increase the chain's temperature so it stops saying "I do not know."

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: re-running manually may not reproduce the production traffic distribution. The inference table HAS the actual failure cases; using them is faster than reproducing.
- B is correct: inference tables are the canonical post-deployment debugging surface. Query the table for the failure pattern, inspect the request, the retrieved chunks (via the trace), and the response. Section 6 of the exam guide names inference logging as the RAG-debugging tool.
- C is wrong: redeploying just to add prints is slow and wastes a deployment.
- D is wrong: temperature changes do not solve "the chain does not have the answer in its corpus"; they make the chain more likely to hallucinate.

</details>
